In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
import matplotlib.pyplot as plt

**2. Load Dataset (CIFAR-10)**

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize
x_train, x_test = x_train / 255.0, x_test / 255.0

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


**3. Load Pre-trained Model (VGG16)**

In [ ]:
base_model = VGG16(weights='imagenet',
                   include_top=False,
                   input_shape=(32,32,3))

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**4. Freeze Base Layers**

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

**5. Add Classifier Head**

In [ ]:
x = base_model.output
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

**6. Compile Model**

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

**7. Train (Frozen Model)**

In [ ]:
history1 = model.fit(x_train, y_train,
                     epochs=5,
                     validation_data=(x_test, y_test))

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 473s 302ms/step - accuracy: 0.5187 - loss: 1.3810 - val_accuracy: 0.5572 - val_loss: 1.2707
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 501s 301ms/step - accuracy: 0.5824 - loss: 1.1928 - val_accuracy: 0.5824 - val_loss: 1.1941
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 498s 299ms/step - accuracy: 0.5996 - loss: 1.1419 - val_accuracy: 0.5850 - val_loss: 1.1758
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 467s 299ms/step - accuracy: 0.6143 - loss: 1.0995 - val_accuracy: 0.5944 - val_loss: 1.1524
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 462s 296ms/step - accuracy: 0.6269 - loss: 1.0686 - val_accuracy: 0.6058 - val_loss: 1.1373


**8. Fine-Tuning (Unfreeze Top Layers)**

In [ ]:
# Unfreeze last few layers
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history2 = model.fit(x_train, y_train,
                     epochs=5,
                     validation_data=(x_test, y_test))

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1445s 924ms/step - accuracy: 0.7282 - loss: 0.7721 - val_accuracy: 0.6940 - val_loss: 0.8850
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1429s 914ms/step - accuracy: 0.7701 - loss: 0.6558 - val_accuracy: 0.7091 - val_loss: 0.8586
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1450s 928ms/step - accuracy: 0.8025 - loss: 0.5645 - val_accuracy: 0.7118 - val_loss: 0.8452
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1442s 922ms/step - accuracy: 0.8317 - loss: 0.4854 - val_accuracy: 0.7172 - val_loss: 0.8358
Epoch 5/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 1462s 935ms/step - accuracy: 0.8565 - loss: 0.4215 - val_accuracy: 0.7228 - val_loss: 0.8422


**9. Evaluate Model**

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Final Accuracy:", test_acc)

310/313 ━━━━━━━━━━━━━━━━━━━━ 0s 252ms/step - accuracy: 0.7267 - loss: 0.8374

**10. Accuracy Comparison Graph**

In [ ]:
plt.plot(history1.history['val_accuracy'], label='Frozen Model')
plt.plot(history2.history['val_accuracy'], label='Fine-tuned Model')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy Improvement')
plt.show()

NameError: name 'plt' is not defined